In [1]:
import lightgbm as lgb
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import GridSearchCV

from sales_dataset import SalesDataset

In [2]:
df_xls = pd.read_excel('temp_8_9_10_итог_с_июлем.xlsx')
ds = SalesDataset(df_xls, start_date='2024-01-01', cn_mes=19)
X, y = ds.load_data_regress()

# df_csv = pd.read_csv('features_sale.csv')
# y = df_csv[['fact_sale']].to_numpy().reshape(-1)
# df_x = df_csv.drop(['fact_sale'], axis=1)
# X = df_x.to_numpy()

TypeError: SalesDataset.__init__() got an unexpected keyword argument 'cn_mes'

In [10]:
# 2. Разделение данных
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Создание датасетов LightGBM
train_data = lgb.Dataset(X_train, label=y_train)
valid_data = lgb.Dataset(X_test, label=y_test, reference=train_data)

In [11]:
# Оценка качества модели
def evaluate_model(y_true, y_pred, model_name):
    """Функция для оценки модели регрессии"""
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    print(f"\n{model_name} Результаты:")
    print(f"Среднеквадратичная ошибка (MSE): {mse:.4f}")
    print(f"Корень из MSE (RMSE): {rmse:.4f}")
    print(f"Средняя абсолютная ошибка (MAE): {mae:.4f}")
    print(f"Коэффициент детерминации (R²): {r2:.4f}")

    return {"MSE": mse, "RMSE": rmse, "MAE": mae, "R2": r2}

**Пример использования ранней остановки**

In [12]:
# Параметры модели
params = {
    'objective': 'regression',
    'metric': 'mae',
    'learning_rate': 0.01,
    'max_depth': 5,
    'num_leaves': 31,
    'verbosity': -1
}

# Обучение с ранней остановкой
model_early_stop = lgb.train(
    params,
    train_data,
    valid_sets=[valid_data],
    num_boost_round=1000,
    callbacks=[
        lgb.early_stopping(stopping_rounds=300), # Если в течение 50 итераций подряд метрика не стала лучше
        lgb.log_evaluation(50)  # Выводим логи каждые 50 итераций
    ]
)

# Сохранение / Загрузка модели
model_early_stop.save_model("lgb_model_regress.txt")
loaded_model = lgb.Booster(model_file="lgb_model_regress.txt")

# Предсказание на тестовых данных
y_pred = loaded_model.predict(X_test)
early_results = evaluate_model(y_test, y_pred, "LightGBM с ранней остановкой")

print(f"\nКоличество использованных деревьев: {model_early_stop.best_iteration}")

Training until validation scores don't improve for 300 rounds
[50]	valid_0's l1: 5.60325
[100]	valid_0's l1: 4.7363
[150]	valid_0's l1: 4.38606
[200]	valid_0's l1: 4.26429
[250]	valid_0's l1: 4.22274
[300]	valid_0's l1: 4.21893
[350]	valid_0's l1: 4.22372
[400]	valid_0's l1: 4.21623
[450]	valid_0's l1: 4.20669
[500]	valid_0's l1: 4.20569
[550]	valid_0's l1: 4.20823
[600]	valid_0's l1: 4.20684
[650]	valid_0's l1: 4.20738
[700]	valid_0's l1: 4.20841
[750]	valid_0's l1: 4.20731
Early stopping, best iteration is:
[495]	valid_0's l1: 4.20527

LightGBM с ранней остановкой Результаты:
Среднеквадратичная ошибка (MSE): 39.4535
Корень из MSE (RMSE): 6.2812
Средняя абсолютная ошибка (MAE): 4.2053
Коэффициент детерминации (R²): 0.6513

Количество использованных деревьев: 495


**Оптимизация гиперпараметров с помощью GridSearchCV**

In [13]:
# Определяем сетку параметров для поиска
param_grid = {
    'n_estimators': [100, 200, 300], # Количество деревьев
    'learning_rate': [0.01, 0.1, 0.2], # Скорость обучения
    'max_depth': [3, 5, 7, 9, 10, 12], # Максимальная глубина деревьев
    'num_leaves': [31, 50, 100], # Количество листьев в дереве
    'subsample': [0.5, 0.8, 1.0] # Доля выборки для обучения каждого дерева
}

# Создаем и обучаем модель с поиском по сетке
grid_search = GridSearchCV(
    estimator=lgb.LGBMRegressor(random_state=42, verbosity=-1),
    param_grid=param_grid,
    cv=3,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=0
)

In [14]:
grid_search.fit(X_train, y_train)

print(f"Лучшие параметры: {grid_search.best_params_}")
print(f"Лучший MSE: {-grid_search.best_score_:.4f}")

# Используем лучшую модель
best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)
best_results = evaluate_model(y_test, y_pred_best, "Оптимизированный LightGBM")

Лучшие параметры: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100, 'num_leaves': 31, 'subsample': 0.5}
Лучший MSE: 38.1297

Оптимизированный LightGBM Результаты:
Среднеквадратичная ошибка (MSE): 40.1627
Корень из MSE (RMSE): 6.3374
Средняя абсолютная ошибка (MAE): 4.2471
Коэффициент детерминации (R²): 0.6450


D:\Virtual_Env_2\prognoz_prodaj_12_2025\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
